# Atlas ARCH-v2: first tests (CIFAR-100, online class-incremental)

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**. Expect roughly 1 to 1.5 hours.
At the end a file `atlas_v2_results.json` downloads. Send that file back.

**What this notebook does, in order**
1. Runs every piece of analysis code against synthetic data with known answers, and against scikit-learn. If any check fails, the notebook stops before touching CIFAR.
2. Runs the audit's design tests (synthetic).
3. Runs a small smoke test of the full pipeline on CIFAR-100 (3 tasks, 15 classes), including a determinism check.
4. Runs the real experiment: 2 configs × 5 seeds, 20 tasks of 5 classes, single online pass. Results are saved after every run, so if Colab disconnects, re-run all and finished runs are skipped.
5. Prints the tables and the pre-registered comparisons, and downloads the JSON.

**Honesty notes (read before interpreting)**
- These are **fresh runs, not a reproduction of the T-S5 runs in ewt.md**. That code was never uploaded. The learner here is my re-implementation (a ~107k-parameter 3-block GroupNorm CNN, ER with reservoir buffer, SGD lr 0.03, batch 10 stream + 10 replay). Absolute numbers can differ from ewt's 24.9 / 30.56 / 35.3. Only comparisons *inside this notebook* are clean.
- All arms are scored by the **same LDA with the same pooled covariance**, so differences come only from the class means (the X3 lesson). A second score with the oracle covariance is reported as a robustness check.
- Class statistics are recorded at the **end of each class's own task** (all its stream items, in that task's final encoder). This is a declared simplification that applies equally to every arm.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # must be set before CUDA starts (determinism)
import sys, json, time, gc, importlib, subprocess, platform
import numpy as np, torch
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '| numpy', np.__version__, '| python', platform.python_version(), '| GPU', GPU)
if GPU is None:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. On CPU this will take many hours.')

## Pre-registration (written before any CIFAR result)

**Arms.** Every arm estimates the 100 class means in the final encoder f_T. Same LDA, same Σ for all.

| Arm | What it is |
|---|---|
| `O_full` | Ceiling: true means of all stream items in f_T (ewt's 35.3 analogue) |
| `B_only_buf` / `B_only_res` | Mean of the class's final-buffer items / its 10 (or 5) reserve-bridge items in f_T |
| `CH_cv_buf` / `CH_cv_res` | X2-style chained transport: each task, one global ridge map old→new fitted on the bridge, applied to every carried mean |
| `GR_prev_*` | Chained per-class regression (GREG): bridge mean of the class + slope × (carried mean − bridge mean in old coordinates) |
| `GR_stable_*` | One hop from the stable random chart r (exact means in r) straight into f_T |
| `GR_multi_*` | ARCH-v2's estimator: chained GREG with auxiliaries [r, f_prev]; r's block gets its own penalty (it can be dropped by CV) |
| `RC_LDA`, `ER_head` | Sanity references only: LDA in r with exact stats (ewt 28.2), the learner's own head (ewt ER 17.4) |

`_buf` = bridge is the replay buffer (items the learner trained on). `_res` = bridge is the held-out reserve (never trained on).

**Configs.** A: buffer M=2000 + reserve 20/class (10 bridge, 10 control). This spends extra bytes on the reserve. B: buffer M=1000 + reserve 10/class (5 + 5): same 2000 stored images as an M=2000 ER baseline.

**Test 1 (R²).** Out-of-fold within-class R² on reserve items of f_T from r, from f_{T-1}, f_{T-5}, f_1, from [r, f_{T-1}], and per hop.

**Comparisons and the rule.** Paired over 5 seeds; significant only if |mean diff| > 2·SE **and** all 5 seeds share the sign. Not significant is not a tie.
- **H1** A: `GR_multi_res` > `CH_cv_buf`. *Falsifier of ARCH-v2's migration estimator.*
- **H2** A: `GR_multi_res` > `GR_multi_buf` (reserve bridge beats buffer bridge). *Falsifier of the reserve-as-bridge choice.*
- **H3** A: `GR_multi_res` > `CH_cv_res` (per-class correction plus r beats a global map, same bridge).
- **H4** equal bytes: B `GR_multi_res` > A `CH_cv_buf`.
- **S1** A: `CH_cv_buf` > `B_only_buf` (does X2's direction replicate here?). **S2** A: `GR_stable_res` > `B_only_res` (does r help at all?).

**My predictions, with honest confidence.** R²(f_T | r) between 0.2 and 0.5, so r is a weak auxiliary (60%). Hop R² ≥ 0.8 (75%). S1 replicates (80%). H3 supported (65%). H1 supported but small, +0.5 to +3 points (55%). H2 a coin flip (50%): the reserve is unbiased but has fewer items per class than the buffer. H4 less likely (40%): halving the buffer may cost the learner more than the estimator gains.

**Decisions fixed on synthetic data before any CIFAR run:** GREG slope fitted on all bridge items (cross-fitting lost badly with 5 items/class); ridge penalty by 5-fold CV over a wide grid; r's block gets a separate penalty from {drop, 0.1, 0.3, 1}.

## 1. Analysis library (pure numpy)

In [ ]:
%%writefile atlas_lib.py
# ===== atlas_lib: pure-numpy analysis code (estimators, LDA, statistics) =====
import numpy as np

def ledoit_wolf(Xc):
    """Ledoit-Wolf shrinkage covariance of already-centred data Xc (n x p).
    Same formula as sklearn.covariance.ledoit_wolf (assume_centered=True)."""
    Xc = np.asarray(Xc, dtype=np.float64)
    n, p = Xc.shape
    X2 = Xc ** 2
    emp_cov_trace = X2.sum(0) / n
    mu = emp_cov_trace.sum() / p
    beta_ = (X2.T @ X2).sum()
    delta_ = ((Xc.T @ Xc) ** 2).sum() / n ** 2
    beta = 1.0 / (p * n) * (beta_ / n - delta_)
    delta = (delta_ - 2.0 * mu * emp_cov_trace.sum() + p * mu ** 2) / p
    beta = min(beta, delta)
    s = 0.0 if beta == 0 else beta / delta
    S = Xc.T @ Xc / n
    return (1 - s) * S + s * mu * np.eye(p), s

def class_center(X, y):
    """Subtract each sample's class mean. Returns centred X and dict of class means."""
    Xc = np.empty_like(X, dtype=np.float64)
    means = {}
    for c in np.unique(y):
        m = y == c
        means[c] = X[m].mean(0)
        Xc[m] = X[m] - means[c]
    return Xc, means

def pooled_cov(X, y):
    Xc, _ = class_center(X, y)
    return ledoit_wolf(Xc)[0]

def lda_predict(X, M, Sigma):
    """Uniform-prior LDA. M: (C x d) class means (rows), Sigma: (d x d). Returns argmax index into rows of M."""
    P = np.linalg.solve(Sigma, M.T)            # d x C  = Sigma^-1 mu_c
    b = -0.5 * np.einsum('dc,cd->c', P, M)     # -1/2 mu^T Sigma^-1 mu
    S = X @ P + b
    return S.argmax(1)

EIGH = np.linalg.eigh   # the notebook may swap in a GPU float64 eigh with identical results (checked in a cell)

def _ridge_factory(Xs, Y):
    """Returns a function lam -> ridge coefficients for standardised Xs, using the smaller Gram matrix."""
    n, p = Xs.shape
    if n >= p:
        ev, V = EIGH(Xs.T @ Xs); ev = np.clip(ev, 0, None)
        VtXtY = V.T @ (Xs.T @ Y)
        return lambda lam: V @ (VtXtY / (ev + lam)[:, None])
    ev, U = EIGH(Xs @ Xs.T); ev = np.clip(ev, 0, None)
    UtY = U.T @ Y
    return lambda lam: Xs.T @ (U @ (UtY / (ev + lam)[:, None]))

def ridge_cv(X, Y, rng, k=5, ngrid=16, split=None, kappas=(0.0, 0.1, 0.3, 1.0)):
    """Ridge regression Y ~ X @ B on centred data (no intercept), penalty chosen by k-fold CV on (X,Y) only.
    Columns of X are standardised internally; returned B is in original units.
    split=p1: columns [:p1] form a separate block (the stable chart r) whose penalty is chosen separately,
    by scaling that block by kappa (kappa=0 drops it). This stops a wide noisy block from forcing one
    shared penalty onto the informative block."""
    X = np.asarray(X, np.float64); Y = np.asarray(Y, np.float64)
    n = X.shape[0]
    sd = X.std(0) + 1e-8
    Xs0 = X / sd
    ks = kappas if split else (1.0,)
    k = max(2, min(k, n))
    folds = rng.permutation(n) % k
    best = (np.inf, None, None)
    for kap in ks:
        scale = np.ones(X.shape[1])
        if split: scale[:split] = kap
        Xs = Xs0 * scale if not (split and kap == 0) else Xs0[:, split:]
        smax2 = np.linalg.norm(Xs, 2) ** 2 + 1e-12
        lams = smax2 * np.logspace(-9, 1, ngrid)
        err = np.zeros(ngrid)
        for f in range(k):
            tr, te = folds != f, folds == f
            coef = _ridge_factory(Xs[tr], Y[tr])
            for jj, lam in enumerate(lams):
                err[jj] += ((Xs[te] @ coef(lam) - Y[te]) ** 2).sum()
        jj = int(err.argmin())
        if err[jj] < best[0]:
            best = (err[jj], kap, lams[jj])
    _, kap, lam = best
    if split and kap == 0:
        B = np.zeros((X.shape[1], Y.shape[1]))
        B[split:] = _ridge_factory(Xs0[:, split:], Y)(lam) / sd[split:, None]
        return B, lam, kap
    scale = np.ones(X.shape[1])
    if split: scale[:split] = kap
    B = _ridge_factory(Xs0 * scale, Y)(lam) * (scale / sd)[:, None]
    return B, lam, kap

def greg_means(classes, mu_aux, A_br, Y_br, y_br, rng, crossfit=False, split=None):
    """Regression (control-variate / GREG) estimate of class means of Y.
    classes : list of class ids to estimate
    mu_aux  : dict class -> population mean of auxiliary features (known or carried)
    A_br, Y_br, y_br : bridge items' auxiliary features, target features, labels
    Estimate: mu_Y,c = mean_B,c(Y) + (mu_aux,c - mean_B,c(A)) @ B, B = pooled within-class slope.
    Default: one pooled slope fitted on all bridge items (crossfit=False). Optional 2-fold cross-fitting;
    on synthetic data it lost badly with small bridges (5 items/class), so it is off (decided before any CIFAR run).
    Classes with no bridge items fall back to a global affine map (flagged).
    Returns dict class -> estimate, set of fallback classes, info."""
    A_br = np.asarray(A_br, np.float64); Y_br = np.asarray(Y_br, np.float64)
    fold = np.zeros(len(y_br), int)
    for c in np.unique(y_br):
        idx = np.where(y_br == c)[0]
        fold[idx[rng.permutation(len(idx))]] = np.arange(len(idx)) % 2
    out, fallback = {}, set()
    present = set(np.unique(y_br).tolist())
    if crossfit:
        Bs = []
        for f in (0, 1):
            tr = fold != f
            Ac, _ = class_center(A_br[tr], y_br[tr]); Yc, _ = class_center(Y_br[tr], y_br[tr])
            Bs.append(ridge_cv(Ac, Yc, rng, split=split)[0])
    Ac_all, _ = class_center(A_br, y_br); Yc_all, _ = class_center(Y_br, y_br)
    B_all = ridge_cv(Ac_all, Yc_all, rng, split=split)[0]
    for c in classes:
        if c not in present:
            fallback.add(c)
            out[c] = Y_br.mean(0) + (mu_aux[c] - A_br.mean(0)) @ B_all
            continue
        if not crossfit:
            m = y_br == c
            out[c] = Y_br[m].mean(0) + (mu_aux[c] - A_br[m].mean(0)) @ B_all
            continue
        est, w = 0.0, 0
        for f in (0, 1):
            m = (y_br == c) & (fold == f)
            if m.sum() == 0:
                continue
            est = est + m.sum() * (Y_br[m].mean(0) + (mu_aux[c] - A_br[m].mean(0)) @ Bs[1 - f])
            w += m.sum()
        out[c] = est / w
    return out, fallback

def global_affine_means(classes, mu_aux, A_br, Y_br, y_br, rng, split=None):
    """X2-style chained map: pooled within-class-centred ridge slope, global intercept."""
    A_br = np.asarray(A_br, np.float64); Y_br = np.asarray(Y_br, np.float64)
    Ac, _ = class_center(A_br, y_br); Yc, _ = class_center(Y_br, y_br)
    B = ridge_cv(Ac, Yc, rng, split=split)[0]
    a0, y0 = A_br.mean(0), Y_br.mean(0)
    return {c: y0 + (mu_aux[c] - a0) @ B for c in classes}

def within_class_r2(A, Y, y, rng, split=None):
    """Out-of-fold within-class R^2 of Y predicted from A (2-fold cross-fit, centring by training-fold class means)."""
    A = np.asarray(A, np.float64); Y = np.asarray(Y, np.float64)
    fold = np.zeros(len(y), int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        fold[idx[rng.permutation(len(idx))]] = np.arange(len(idx)) % 2
    sse = sst = 0.0
    for f in (0, 1):
        tr, te = fold != f, fold == f
        Ac, ma = class_center(A[tr], y[tr]); Yc, my = class_center(Y[tr], y[tr])
        B = ridge_cv(Ac, Yc, rng, split=split)[0]
        for c in np.unique(y[te]):
            if c not in ma:
                continue
            m = te & (y == c)
            pred = my[c] + (A[m] - ma[c]) @ B
            sse += ((Y[m] - pred) ** 2).sum()
            sst += ((Y[m] - my[c]) ** 2).sum()
    return 1 - sse / sst

def paired(a, b):
    """Pre-registered rule: diff = a-b over seeds; significant iff |mean| > 2*SE and all seeds share the sign."""
    d = np.asarray(a, float) - np.asarray(b, float)
    n = len(d)
    se = d.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan')
    sig = bool(n > 1 and abs(d.mean()) > 2 * se and (np.all(d > 0) or np.all(d < 0)))
    return dict(mean=float(d.mean()), se=float(se), n_pos=int((d > 0).sum()), n=n, sig=sig)

def spearman(a, b):
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1])

In [ ]:
%%writefile atlas_analysis.py
# ===== atlas_analysis: everything after the features exist (pure numpy) =====
import numpy as np, time

def run_analysis(D, seed):
    """D: dict with
         tasks       list of arrays of class ids (task t = tasks[t])
         y_train     labels of all train items
         stream_ids  train ids the learner saw (excludes the reserve)
         bridge_ids, control_ids   reserve splits (never trained on)
         F           list of T arrays, F[t] = features of ALL train items under snapshot f_t (end of task t)
         R           stable random-chart features of all train items
         buffers     list of T arrays, train ids in the replay buffer at the end of task t
         Ftest, Rtest, y_test     test features under the final f_T, under r, and labels
         head_acc    accuracy of the learner's own softmax head (sanity only)
    Returns a dict of results for this run."""
    rng = np.random.default_rng(10_000 + seed)
    t0 = time.time()
    tasks = [np.asarray(c) for c in D['tasks']]; T = len(tasks)
    classes = np.concatenate(tasks)
    y = np.asarray(D['y_train']); F = D['F']; R = D['R']
    stream = np.asarray(D['stream_ids']); br = np.asarray(D['bridge_ids']); ct = np.asarray(D['control_ids'])
    res_ids = np.concatenate([br, ct])
    task_of = {int(c): t for t, cs in enumerate(tasks) for c in cs}
    # ---- leakage / integrity asserts ----
    assert len(np.intersect1d(stream, res_ids)) == 0, 'reserve leaked into stream'
    assert len(np.intersect1d(br, ct)) == 0
    for b in D['buffers']:
        assert np.isin(b, stream).all(), 'buffer holds a non-stream item'
    assert set(np.unique(y[res_ids]).tolist()) == set(classes.tolist())
    f64 = lambda a: np.asarray(a, np.float64)
    ids_of = {int(c): stream[y[stream] == c] for c in classes}
    FT = F[T - 1]
    # ---- exact statistics ----
    mu_r = {c: f64(R[ids]).mean(0) for c, ids in ids_of.items()}                 # stable chart, all confirmed data
    mu_rec = {c: f64(F[task_of[c]][ids]).mean(0) for c, ids in ids_of.items()}   # recorded at end of own task
    oracle = {c: f64(FT[ids]).mean(0) for c, ids in ids_of.items()}              # ceiling: all stream data in f_T
    Sig_shared = pooled = None
    from atlas_lib import pooled_cov, lda_predict, greg_means, global_affine_means, within_class_r2
    Sig_shared = pooled_cov(f64(FT[res_ids]), y[res_ids])                        # same Sigma for every arm
    Sig_oracle = pooled_cov(f64(FT[stream]), y[stream])
    Sinv = np.linalg.inv(Sig_shared)
    ytest = np.asarray(D['y_test']); keep = np.isin(ytest, classes)
    Ftest = f64(D['Ftest'])[keep]; Rtest = f64(D['Rtest'])[keep]; ytest = ytest[keep]
    old_cls = set(np.concatenate(tasks[: T // 2]).tolist())

    def evaluate(means, Sig=Sig_shared, Xte=Ftest):
        M = np.stack([means[int(c)] for c in classes])
        return float((classes[lda_predict(Xte, M, Sig)] == ytest).mean() * 100)

    def mean_err(means, subset=None):
        cs = [int(c) for c in classes if subset is None or int(c) in subset]
        d = np.stack([means[c] - oracle[c] for c in cs])
        return float(np.einsum('nd,de,ne->n', d, Sinv, d).mean())

    # ---- arms ----
    arms, fallbacks = {}, {}
    arms['O_full'] = oracle

    def bonly(ids_pool):
        g = f64(FT[ids_pool]).mean(0); out = {}; nfb = 0
        for c in classes:
            m = ids_pool[y[ids_pool] == c]
            if len(m) == 0: out[int(c)] = g; nfb += 1
            else: out[int(c)] = f64(FT[m]).mean(0)
        return out, nfb
    arms['B_only_buf'], fallbacks['B_only_buf'] = bonly(np.asarray(D['buffers'][T - 1]))
    arms['B_only_res'], fallbacks['B_only_res'] = bonly(br)

    def bridge_at(t_new, kind):
        """bridge items available when hopping into f_{t_new}"""
        if kind == 'buf':
            return np.asarray(D['buffers'][t_new])
        seen = np.concatenate(tasks[: t_new + 1])
        return br[np.isin(y[br], seen)]

    def chain(kind, bkind):
        est = {int(c): mu_rec[int(c)] for c in tasks[0]}; nfb = 0
        for t in range(T - 1):
            old = [int(c) for c in np.concatenate(tasks[: t + 1])]
            ids = bridge_at(t + 1, bkind)
            A = f64(F[t][ids]); Y = f64(F[t + 1][ids]); yb = y[ids]
            if kind == 'multi':
                A = np.hstack([f64(R[ids]), A])
                mu_aux = {c: np.concatenate([mu_r[c], est[c]]) for c in old}
            else:
                mu_aux = est
            split = R.shape[1] if kind == 'multi' else None
            if kind == 'aff':
                est = global_affine_means(old, mu_aux, A, Y, yb, rng)
            else:
                est, fb = greg_means(old, mu_aux, A, Y, yb, rng, split=split); nfb += len(fb)
            for c in tasks[t + 1]:
                est[int(c)] = mu_rec[int(c)]
        return est, nfb

    for kind, name in (('aff', 'CH_cv'), ('prev', 'GR_prev'), ('multi', 'GR_multi')):
        for bk in ('buf', 'res'):
            arms[f'{name}_{bk}'], fallbacks[f'{name}_{bk}'] = chain(kind, bk)
    for bk in ('buf', 'res'):
        ids = bridge_at(T - 1, bk)
        est, fb = greg_means([int(c) for c in classes], mu_r, f64(R[ids]), f64(FT[ids]), y[ids], rng)
        arms[f'GR_stable_{bk}'], fallbacks[f'GR_stable_{bk}'] = est, len(fb)

    out = {'acc': {}, 'acc_oracleSigma': {}, 'err_all': {}, 'err_old': {}, 'err_recent': {}, 'fallbacks': fallbacks}
    recent = set(classes.tolist()) - old_cls
    for k, m in arms.items():
        out['acc'][k] = evaluate(m)
        out['acc_oracleSigma'][k] = evaluate(m, Sig_oracle)
        out['err_all'][k] = mean_err(m); out['err_old'][k] = mean_err(m, old_cls); out['err_recent'][k] = mean_err(m, recent)
    # sanity references (not part of the hypotheses)
    Sig_r = pooled_cov(f64(R[stream]), y[stream])
    out['acc']['RC_LDA'] = evaluate(mu_r, Sig_r, Rtest)
    out['acc']['ER_head'] = float(D['head_acc'])
    # ---- Test 1: within-class R^2 on reserve items (never trained on) ----
    r2 = {}
    Yr = f64(FT[res_ids]); yr = y[res_ids]
    r2['fT|r'] = within_class_r2(f64(R[res_ids]), Yr, yr, rng)
    for lag in (1, 5, T - 1):
        if 1 <= lag <= T - 1:
            r2[f'fT|f(T-{lag})'] = within_class_r2(f64(F[T - 1 - lag][res_ids]), Yr, yr, rng)
    if T >= 2:
        r2['fT|[r,f(T-1)]'] = within_class_r2(np.hstack([f64(R[res_ids]), f64(F[T - 2][res_ids])]), Yr, yr, rng, split=R.shape[1])
        hops = [within_class_r2(f64(F[t][res_ids]), f64(F[t + 1][res_ids]), yr, rng) for t in range(T - 1)]
        r2['hop_mean'] = float(np.mean(hops)); r2['hop_min'] = float(np.min(hops)); r2['hops'] = [float(h) for h in hops]
    out['r2'] = {k: (float(v) if not isinstance(v, list) else v) for k, v in r2.items()}
    out['seconds'] = time.time() - t0
    return out

In [ ]:
%%writefile atlas_summary.py
# ===== Summary and pre-registered hypothesis tests =====
import numpy as np
from atlas_lib import paired

HYPOTHESES = [  # (name, (config, arm), (config, arm), what a positive significant difference means)
    ('H1 multi-aux GREG (reserve bridge) beats X2-style CH_cv (buffer bridge)', ('A', 'GR_multi_res'), ('A', 'CH_cv_buf')),
    ('H2 reserve bridge beats buffer bridge, same estimator',                    ('A', 'GR_multi_res'), ('A', 'GR_multi_buf')),
    ('H3 per-class GREG with [r,f_prev] beats CH_cv, same reserve bridge',       ('A', 'GR_multi_res'), ('A', 'CH_cv_res')),
    ('H4 equal bytes: B multi-aux (M=1000+1000 reserve) beats A CH_cv_buf (M=2000)', ('B', 'GR_multi_res'), ('A', 'CH_cv_buf')),
    ('S1 replication of X2 direction: CH_cv_buf beats buffer-only',              ('A', 'CH_cv_buf'), ('A', 'B_only_buf')),
    ('S2 stable chart helps at all: GR_stable_res beats reserve-only means',     ('A', 'GR_stable_res'), ('A', 'B_only_res')),
]

def table(results, cfg, metric='acc'):
    runs = results[cfg]; seeds = sorted(runs)
    arms = list(runs[seeds[0]][metric].keys())
    rows = []
    for a in arms:
        v = np.array([runs[s][metric][a] for s in seeds])
        rows.append((a, v.mean(), v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else float('nan')))
    return sorted(rows, key=lambda r: -r[1] if metric.startswith('acc') else r[1]), seeds

def summarize(results, print_fn=print):
    out = {'tables': {}, 'hypotheses': [], 'r2': {}}
    for cfg in sorted(results):
        for metric in ('acc', 'acc_oracleSigma', 'err_all', 'err_old'):
            rows, seeds = table(results, cfg, metric)
            out['tables'][f'{cfg}/{metric}'] = [(a, float(m), float(s)) for a, m, s in rows]
            if metric in ('acc', 'err_all'):
                print_fn(f'\nConfig {cfg}, {metric}  (mean ± SE over seeds {seeds})')
                for a, m, s in rows: print_fn(f'   {a:16s} {m:8.3f} ± {s:.3f}')
        seeds = sorted(results[cfg]); keys = [k for k in results[cfg][seeds[0]]['r2'] if k != 'hops']
        out['r2'][cfg] = {k: (float(np.mean([results[cfg][s]['r2'][k] for s in seeds])),
                              float(np.std([results[cfg][s]['r2'][k] for s in seeds], ddof=1) / np.sqrt(len(seeds))) if len(seeds) > 1 else float('nan'))
                          for k in keys}
        print_fn(f'\nConfig {cfg}, Test 1 within-class R² on reserve items:')
        for k, (m, s) in out['r2'][cfg].items(): print_fn(f'   {k:16s} {m:.3f} ± {s:.3f}')
    print_fn('\nPre-registered comparisons (accuracy, shared Σ). Significant = |mean| > 2·SE and all seeds same sign:')
    for name, (ca, aa), (cb, ab) in HYPOTHESES:
        if ca not in results or cb not in results: continue
        seeds = sorted(set(results[ca]) & set(results[cb]))
        r = paired([results[ca][s]['acc'][aa] for s in seeds], [results[cb][s]['acc'][ab] for s in seeds])
        r2 = paired([results[ca][s]['acc_oracleSigma'][aa] for s in seeds], [results[cb][s]['acc_oracleSigma'][ab] for s in seeds])
        verdict = ('SUPPORTED' if r['mean'] > 0 else 'REVERSED') if r['sig'] else 'NOT SIGNIFICANT (not a tie)'
        out['hypotheses'].append(dict(name=name, a=f'{ca}/{aa}', b=f'{cb}/{ab}', shared_sigma=r, oracle_sigma=r2, verdict=verdict))
        print_fn(f'  {name}\n     diff {r["mean"]:+.2f} ± {r["se"]:.2f} ({r["n_pos"]}/{r["n"]} seeds positive) -> {verdict};'
                 f'  with oracle Σ: {r2["mean"]:+.2f} ± {r2["se"]:.2f}')
    return out

In [ ]:
%%writefile atlas_torch.py
# ===== Learner, stable random chart, online stream, feature extraction (PyTorch) =====
import os, copy, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as Fn, torchvision
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
MEAN = torch.tensor([0.5071, 0.4865, 0.4409], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.2673, 0.2564, 0.2762], device=DEV).view(1, 3, 1, 1)

def load_cifar100(root='/content/data'):
    tr = torchvision.datasets.CIFAR100(root, train=True, download=True)
    te = torchvision.datasets.CIFAR100(root, train=False, download=True)
    to_t = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).contiguous().to(DEV)   # uint8 N,3,32,32 on GPU
    Xtr, ytr, Xte, yte = to_t(tr.data), np.array(tr.targets), to_t(te.data), np.array(te.targets)
    assert Xtr.shape == (50000, 3, 32, 32) and Xte.shape == (10000, 3, 32, 32)
    assert (np.bincount(ytr) == 500).all() and (np.bincount(yte) == 100).all()
    return Xtr, ytr, Xte, yte

def to_input(xu8):
    return (xu8.float() / 255.0 - MEAN) / STD

def make_encoder():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.GroupNorm(8, 32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.ReLU())

def embed(enc, x):
    return enc(x).mean((2, 3))           # global average pool (plain mean: deterministic backward)

class Net(nn.Module):
    def __init__(self):
        super().__init__(); self.enc = make_encoder(); self.head = nn.Linear(128, 100)
    def forward(self, x):
        return self.head(embed(self.enc, x))

class RandomChart:
    """Stable reference r: fixed random 5x5 conv (256 maps) + ReLU + 2x2 average pool -> 1024-d. Never trained."""
    def __init__(self, seed, width=256):
        g = torch.Generator().manual_seed(seed)
        self.W = (torch.randn(width, 3, 5, 5, generator=g) / np.sqrt(75.0)).to(DEV)
        self.b = (0.1 * torch.randn(width, generator=g)).to(DEV)
    def __call__(self, x):
        return Fn.adaptive_avg_pool2d(Fn.relu(Fn.conv2d(x, self.W, self.b, padding=2)), 2).flatten(1)

@torch.no_grad()
def extract(fn, Xu8, bs=2500):
    return torch.cat([fn(to_input(Xu8[i:i + bs])).float().cpu() for i in range(0, len(Xu8), bs)]).numpy()

def run_one(seed, cfg, data, log=print):
    """One online class-incremental run. Returns the dict D that run_analysis expects."""
    Xtr, ytr, Xte, yte = data
    t0 = time.time()
    rng = np.random.default_rng(seed)
    order = rng.permutation(100)[: cfg['n_classes']]
    tasks = order.reshape(cfg['n_tasks'], -1)
    nb = cfg['reserve_per_class'] // 2
    bridge, control, stream_by_class = [], [], {}
    for c in order:
        ids = rng.permutation(np.where(ytr == c)[0])
        bridge.append(ids[:nb]); control.append(ids[nb:2 * nb])
        s = ids[2 * nb:]
        if cfg.get('stream_per_class'): s = s[: cfg['stream_per_class']]
        stream_by_class[int(c)] = s
    bridge, control = np.concatenate(bridge), np.concatenate(control)
    stream_ids = np.concatenate([stream_by_class[int(c)] for c in order])
    assert len(np.intersect1d(stream_ids, np.concatenate([bridge, control]))) == 0
    torch.manual_seed(seed)
    model = Net().to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=cfg['lr'])
    ytr_t = torch.from_numpy(ytr).to(DEV)
    M, B = cfg['M'], cfg['batch']
    buf = np.zeros(M, dtype=np.int64); nbuf = 0; nseen = 0
    buffers, snaps = [], []
    for t, cs in enumerate(tasks):
        ids = rng.permutation(np.concatenate([stream_by_class[int(c)] for c in cs]))
        for i in range(0, len(ids), B):
            bi = ids[i:i + B]
            allidx = np.concatenate([bi, buf[rng.integers(0, nbuf, size=B)]]) if nbuf > 0 else bi  # replay drawn before insertion
            it = torch.from_numpy(allidx).to(DEV)
            loss = Fn.cross_entropy(model(to_input(Xtr[it])), ytr_t[it])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            for j in bi:                                # reservoir sampling (Algorithm R)
                nseen += 1
                if nbuf < M: buf[nbuf] = j; nbuf += 1
                else:
                    k = rng.integers(0, nseen)
                    if k < M: buf[k] = j
            if not torch.isfinite(loss): raise RuntimeError(f'non-finite loss at task {t}')
        buffers.append(buf[:nbuf].copy())
        snaps.append(copy.deepcopy(model.enc).eval())
    t_train = time.time() - t0
    model.eval()
    with torch.no_grad():
        keep = np.isin(yte, order)
        logits = torch.cat([model(to_input(Xte[i:i + 2500])) for i in range(0, len(Xte), 2500)]).cpu().numpy()[keep]
        head_acc = float((order[logits[:, order].argmax(1)] == yte[keep]).mean() * 100)
    F = [extract(lambda x, e=e: embed(e, x), Xtr) for e in snaps]
    chart = RandomChart(seed + 777)
    R, Rtest = extract(chart, Xtr, bs=500), extract(chart, Xte, bs=500)   # 500 keeps the 256x32x32 maps small
    Ftest = extract(lambda x: embed(snaps[-1], x), Xte)
    for a in F + [R, Rtest, Ftest]:
        assert np.isfinite(a).all(), 'non-finite features'
    log(f'  seed {seed}: train {t_train:.0f}s, total {time.time()-t0:.0f}s, ER head acc {head_acc:.1f}, '
        f'buffer {nbuf}, per-class buffer min {np.bincount(ytr[buf[:nbuf]], minlength=100)[order].min()}')
    return dict(tasks=[np.array(c) for c in tasks], y_train=ytr, stream_ids=stream_ids, bridge_ids=bridge,
                control_ids=control, F=F, R=R, buffers=buffers, Ftest=Ftest, Rtest=Rtest, y_test=yte, head_acc=head_acc)

In [ ]:
sys.path.insert(0, '.')
import atlas_lib, atlas_analysis, atlas_summary, atlas_torch
for m in (atlas_lib, atlas_analysis, atlas_summary, atlas_torch): importlib.reload(m)
from atlas_analysis import run_analysis
from atlas_summary import summarize
from atlas_torch import load_cifar100, run_one

## 2. Checks with known answers (the notebook stops here if anything fails)

In [ ]:
%%writefile test_lib.py
import numpy as np, sys
sys.path.insert(0,'.')
from atlas_lib import *
rng=np.random.default_rng(0)
fails=[]
def check(name, cond, info=''):
    print(('PASS' if cond else 'FAIL'), name, info); 
    if not cond: fails.append(name)

# L1 Ledoit-Wolf: shrinkage in [0,1]; identity data -> near mu*I; large n -> near sample cov
X=rng.normal(size=(5000,10)); C,s=ledoit_wolf(X-X.mean(0)); check('LW_large_n', np.abs(C-np.cov(X.T,bias=True)).max()<0.05 and 0<=s<=1, f's={s:.3f}')
X=rng.normal(size=(20,50)); C,s=ledoit_wolf(X-X.mean(0)); check('LW_p>n_wellcond', np.linalg.cond(C)<1e3 and s>0.3, f's={s:.3f}')

# L2 LDA equals brute-force Gaussian log-likelihood argmax
d,Cn=8,5; M=rng.normal(size=(Cn,d))*2; A=rng.normal(size=(d,d)); Sig=A@A.T+np.eye(d)
X=rng.normal(size=(300,d))
Si=np.linalg.inv(Sig); ll=np.stack([-0.5*np.einsum('nd,de,ne->n',X-M[c],Si,X-M[c]) for c in range(Cn)],1)
check('LDA_bruteforce', np.array_equal(lda_predict(X,M,Sig), ll.argmax(1)))

# L3 LDA accuracy on well separated Gaussians ~ Bayes
L=np.repeat(np.arange(Cn),400); Z=M[L]+rng.multivariate_normal(np.zeros(d),Sig,size=len(L))
acc=(lda_predict(Z,M,Sig)==L).mean(); check('LDA_acc_sane', acc>0.5, f'acc={acc:.3f}')

# L4 GREG exact under exact linear relation, no noise (truth recovered)
da,dy,Cn=6,4,10; W=rng.normal(size=(da,dy)); off=rng.normal(size=(Cn,dy))*0  # pure linear: y = a@W + b
b=rng.normal(size=dy)
mu_a={c:rng.normal(size=da)*3 for c in range(Cn)}
yb=np.repeat(np.arange(Cn),12); Ab=np.stack([mu_a[c] for c in yb])+rng.normal(size=(len(yb),da)); Yb=Ab@W+b
est,fb=greg_means(list(range(Cn)),mu_a,Ab,Yb,yb,rng)
err=max(np.abs(est[c]-(mu_a[c]@W+b)).max() for c in range(Cn)); check('GREG_exact_linear', err<1e-3, f'maxerr={err:.2e}')

# L5 GREG variance reduction ~ (1-R2) (Monte Carlo, scalar target, known population means)
res={}
for R2 in (0.0,0.8):
    errs_g=[];errs_b=[]
    for t in range(300):
        N=2000; a=rng.normal(size=(N,3)); e=rng.normal(size=N)
        w=np.array([1.,0.5,-0.5]); w=w/np.linalg.norm(w)*np.sqrt(R2)
        yv=(a@w+np.sqrt(1-R2)*e)[:,None]
        idx=rng.choice(N,30,replace=False)
        est,_=greg_means([0],{0:a.mean(0)},a[idx],yv[idx],np.zeros(30,int),rng)
        errs_g.append(est[0][0]-yv.mean()); errs_b.append(yv[idx].mean()-yv.mean())
    res[R2]=(np.var(errs_g)/np.var(errs_b))
check('GREG_varratio_R2=0', 0.8<res[0.0]<1.35, f'ratio={res[0.0]:.2f} (expect ~1)')
check('GREG_varratio_R2=0.8', res[0.8]<0.4, f'ratio={res[0.8]:.2f} (expect ~0.2 + slope noise)')

# L6 fallback path flagged for classes absent from bridge
est,fb=greg_means([0,1,99],{0:np.zeros(da),1:np.ones(da),99:np.ones(da)},Ab,Yb,yb,rng)
check('GREG_fallback_flag', fb=={99})

# L7 chained exact linear hops recover truth (two hops, invertible maps, exact)
mu0={c:rng.normal(size=5)*3 for c in range(6)}
W1=rng.normal(size=(5,5)); W2=rng.normal(size=(5,5))
yb=np.repeat(np.arange(6),10); F0=np.stack([mu0[c] for c in yb])+rng.normal(size=(60,5))
F1=F0@W1+1.0; F2=F1@W2-2.0
m1,_=greg_means(list(range(6)),mu0,F0,F1,yb,rng); m2,_=greg_means(list(range(6)),m1,F1,F2,yb,rng)
err=max(np.abs(m2[c]-((mu0[c]@W1+1)@W2-2)).max() for c in range(6)); check('chain_exact', err<1e-3, f'{err:.1e}')
g=global_affine_means(list(range(6)),mu0,F0,F1,yb,rng); err=max(np.abs(g[c]-(mu0[c]@W1+1)).max() for c in range(6)); check('global_affine_exact', err<1e-3, f'{err:.1e}')

# L8 within-class R2: ~1 for exact linear, ~0 for independent
A=rng.normal(size=(400,6)); yy=np.repeat(np.arange(20),20)
check('R2_linear', within_class_r2(A,A@rng.normal(size=(6,3)),yy,rng)>0.99)
r0=within_class_r2(A,rng.normal(size=(400,3)),yy,rng); check('R2_indep', abs(r0)<0.1, f'{r0:.3f}')

# L9 paired rule
check('paired_sig', paired([2,2.1,1.9,2.2,2.0],[0,0,0,0,0])['sig'] and not paired([1,-1,1,1,1],[0]*5)['sig'])
# L10 ridge equals closed form both regimes
for n,p in ((200,30),(30,200)):
    X=rng.normal(size=(n,p)); Y=rng.normal(size=(n,3))
    from atlas_lib import _ridge_factory
    lam=3.7; B=_ridge_factory(X,Y)(lam); Bc=np.linalg.solve(X.T@X+lam*np.eye(p),X.T@Y)
    check(f'ridge_closed_form_{n}x{p}', np.abs(B-Bc).max()<1e-8)
# L11 block ridge: kappa=0 path equals plain ridge on the second block; useless wide block is dropped or shrunk
Xa=rng.normal(size=(300,8)); Yb=Xa@rng.normal(size=(8,4))+0.1*rng.normal(size=(300,4))
Xn=rng.normal(size=(300,200))
Bb,lamb,kap=ridge_cv(np.hstack([Xn,Xa]),Yb,np.random.default_rng(1),split=200)
r2_block=1-((np.hstack([Xn,Xa])@Bb-Yb)**2).sum()/(Yb**2).sum()
check('block_ridge_drops_noise', kap<=0.1 and r2_block>0.95, f'kappa={kap} R2={r2_block:.3f}')
Bz=np.zeros((208,4)); 
B0,lam0,_=ridge_cv(Xa,Yb,np.random.default_rng(1))
check('block_ridge_shape', Bb.shape==(208,4))
print('FAILS:',fails)

import sys as _s
if fails: _s.exit(1)

In [ ]:
%%writefile test_analysis.py
import numpy as np, sys; sys.path.insert(0,'.')
from atlas_analysis import run_analysis
fails=[]
def check(n,c,i=''):
    print('PASS' if c else 'FAIL', n, i)
    if not c: fails.append(n)
def synth(noise, seed=0, T=5, per=60, nres=10, M=200, dz=12, d=12, dr=40):
    rng=np.random.default_rng(seed); C=T*3
    order=rng.permutation(C); tasks=list(order.reshape(T,3))
    mz=rng.normal(size=(C,dz))*1.2
    y=np.repeat(np.arange(C),per+nres); Z=mz[y]+rng.normal(size=(len(y),dz))
    yte=np.repeat(np.arange(C),30); Zte=mz[yte]+rng.normal(size=(len(yte),dz))
    W=rng.normal(size=(dz,d)); F=[];
    Ws=[]
    for t in range(T):
        W=W+0.4*rng.normal(size=W.shape); Ws.append(W.copy())
        F.append((Z@W+0.3*t+noise*rng.normal(size=(len(y),d))).astype(np.float32))
    Wr=rng.normal(size=(dz,dr)); R=(np.tanh(Z@Wr*0.5)+0.05*rng.normal(size=(len(y),dr))).astype(np.float32)
    Rte=np.tanh(Zte@Wr*0.5); Fte=Zte@Ws[-1]+0.3*(T-1)+noise*rng.normal(size=(len(yte),d))
    br=[];ctl=[];st=[]
    for c in range(C):
        ids=np.where(y==c)[0]; ids=rng.permutation(ids)
        br+=list(ids[:nres//2]); ctl+=list(ids[nres//2:nres]); st+=list(ids[nres:])
    st=np.array(st); bufs=[]; buf=np.array([],int)
    for t in range(T):
        seen=st[np.isin(y[st],np.concatenate(tasks[:t+1]))]
        buf=rng.choice(seen,min(M,len(seen)),replace=False); bufs.append(buf)
    return dict(tasks=tasks,y_train=y,stream_ids=st,bridge_ids=np.array(br),control_ids=np.array(ctl),F=F,R=R,
                buffers=bufs,Ftest=Fte,Rtest=Rte,y_test=yte,head_acc=0.0)
if __name__=="__main__":
    # exact linear features, enough bridge items: every transport arm must recover the oracle
    o=run_analysis(synth(0.0, nres=40),0)
    for k in ('CH_cv_buf','CH_cv_res','GR_prev_buf','GR_prev_res','GR_multi_buf','GR_multi_res'):
        check('exact_'+k, o['err_all'][k]<1e-3, f"err={o['err_all'][k]:.2e} acc={o['acc'][k]:.1f} oracle={o['acc']['O_full']:.1f}")
    check('exact_hopR2', o['r2']['hop_min']>0.999, o['r2']['hop_min'])
    check('exact_fallbacks_zero', sum(o['fallbacks'].values())==0)
    o=run_analysis(synth(1.0),0)
    for k,v in sorted(o['err_all'].items(), key=lambda kv: kv[1]): print(f'   {k:14s} err={v:.4f} acc={o["acc"][k]:.1f}')
    check('noisy_oracle_err0', o['err_all']['O_full']==0)
    check('noisy_greg_beats_bonly', o['err_all']['GR_prev_buf']<o['err_all']['B_only_buf'])
    check('noisy_greg_res_beats_bonly_res', o['err_all']['GR_multi_res']<o['err_all']['B_only_res'])
    check('block_R2_not_worse', o['r2']['fT|[r,f(T-1)]'] >= o['r2']['fT|f(T-1)'] - 0.02, (o['r2']['fT|[r,f(T-1)]'], o['r2']['fT|f(T-1)']))
    print('FAILS',fails)
    if fails: sys.exit(1)

In [ ]:
%%writefile sk_check.py
# ===== Cross-check our numpy code against scikit-learn =====
import numpy as np, sys; sys.path.insert(0, '.')
import atlas_lib
from sklearn.covariance import ledoit_wolf as sk_lw
from sklearn.linear_model import Ridge
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
rng = np.random.default_rng(7)
for n, p in ((300, 40), (30, 80)):
    X = rng.normal(size=(n, p)) @ rng.normal(size=(p, p)); Xc = X - X.mean(0)
    C1, s1 = atlas_lib.ledoit_wolf(Xc); C2, s2 = sk_lw(Xc, assume_centered=True)
    assert np.allclose(C1, C2, atol=1e-8 * np.abs(C2).max()) and abs(s1 - s2) < 1e-10, ('LW mismatch', n, p)
    Y = rng.normal(size=(n, 3)); lam = 2.5
    B1 = atlas_lib._ridge_factory(X, Y)(lam); B2 = Ridge(alpha=lam, fit_intercept=False, solver='svd').fit(X, Y).coef_.T
    assert np.allclose(B1, B2, atol=1e-6 * np.abs(B2).max()), ('ridge mismatch', n, p)
K, d = 7, 10
yl = np.repeat(np.arange(K), 60); Xl = rng.normal(size=(K, d))[yl] * 1.5 + rng.normal(size=(len(yl), d)) @ rng.normal(size=(d, d)) * 0.5
lda = LinearDiscriminantAnalysis(solver='lsqr', priors=np.ones(K) / K, store_covariance=True).fit(Xl, yl)
Xt = rng.normal(size=(2000, d)) * 2
agree = (atlas_lib.lda_predict(Xt, lda.means_, lda.covariance_) == lda.predict(Xt)).mean()
assert agree > 0.999, ('LDA mismatch', agree)
print('sklearn cross-check PASS: Ledoit-Wolf, ridge, LDA (agreement %.4f)' % agree)

In [ ]:
%%writefile audit_tests.py
# ===== Audit design tests (synthetic, known answers). Each checks a property ARCH-v2 relies on. =====
import numpy as np, sys; sys.path.insert(0, '.')
from atlas_lib import greg_means, global_affine_means, lda_predict, pooled_cov
rng = np.random.default_rng(123)
afails = []
def acheck(n, c, i=''):
    print('PASS' if c else 'FAIL', n, i)
    if not c: afails.append(n)

# U1 Re-anchoring does not reset inherited error: an exact hop carries the old error forward as delta @ W.
d, C = 8, 6
mu0 = {c: rng.normal(size=d) * 3 for c in range(C)}
W = rng.normal(size=(d, d)); b = rng.normal(size=d)
yb = np.repeat(np.arange(C), 30); A = np.stack([mu0[c] for c in yb]) + rng.normal(size=(len(yb), d)); Y = A @ W + b
delta = {c: rng.normal(size=d) * 0.5 for c in range(C)}
carried = {c: mu0[c] + delta[c] for c in range(C)}
est, _ = greg_means(list(range(C)), carried, A, Y, yb, rng)
err_after = np.array([np.linalg.norm(est[c] - (mu0[c] @ W + b)) for c in range(C)])
pred_err = np.array([np.linalg.norm(delta[c] @ W) for c in range(C)])
acheck('U1_inherited_error_persists', np.allclose(err_after, pred_err, rtol=1e-3) and err_after.min() > 0.1,
       f'after={err_after.round(3)} predicted={pred_err.round(3)}')

# U2 A rotation (no shift) is not fixed by a mean shift; a full linear transport fixes it.
Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
Yr = A @ Q
shift = {c: carried_c + (Yr.mean(0) - A.mean(0)) for c, carried_c in mu0.items()}
full = global_affine_means(list(range(C)), mu0, A, Yr, yb, rng)
e_shift = max(np.linalg.norm(shift[c] - mu0[c] @ Q) for c in range(C)); e_full = max(np.linalg.norm(full[c] - mu0[c] @ Q) for c in range(C))
acheck('U2_rotation_needs_full_transport', e_shift > 1.0 and e_full < 1e-3, f'shift_err={e_shift:.2f} full_err={e_full:.1e}')

# U3 Reused evidence: fusing two estimates built from the SAME items as if independent halves the claimed
#    variance but not the real one. The design rule (never count reused evidence as new) is needed.
n, trials = 20, 4000
fused = []
for _ in range(trials):
    x = rng.normal(size=n); e1 = x.mean(); e2 = x.mean() + 0.0  # second "source" re-reads the same items
    fused.append((e1 + e2) / 2)
claimed = 1 / (n + n); real = np.var(fused)
acheck('U3_reused_evidence_overconfident', real / claimed > 1.8, f'real/claimed variance={real/claimed:.2f} (expect 2)')

# U4 A dormant class whose items fall inside active classes' support is still retrieved by the stable chart.
C2, dd = 50, 32
M = rng.normal(size=(C2, dd)) * 0.6       # crowded: classes overlap
Sig = np.eye(dd)
dormant = np.arange(10, 50)
Xq = np.repeat(dormant, 40); Zq = M[Xq] + rng.normal(size=(len(Xq), dd))
P = np.linalg.solve(Sig, M.T); S = Zq @ P - 0.5 * np.einsum('dc,cd->c', P, M)
top5 = np.argsort(-S, 1)[:, :5]; recall = (top5 == Xq[:, None]).any(1).mean()
acheck('U4_dormant_retrieved_top5', recall > 0.5, f'top-5 recall={recall:.2f} (chance 0.10)')

# U5 A label-only change (classes 0 and 1 swap labels) is caught by prequential error, not by input novelty.
C3 = 10; M3 = rng.normal(size=(C3, dd)) * 3
def novelty(Z): return np.min(((Z[:, None, :] - M3[None]) ** 2).sum(-1), 1)
lab = rng.integers(0, C3, 4000); Z = M3[lab] + rng.normal(size=(4000, dd))
before, after = slice(0, 2000), slice(2000, 4000)
obs = lab.copy(); sw = (np.arange(4000) >= 2000) & np.isin(lab, [0, 1]); obs[sw] = 1 - lab[sw]
pred = lda_predict(Z, M3, np.eye(dd))
err_b = (pred[before] != obs[before])[np.isin(obs[before], [0, 1])].mean(); err_a = (pred[after] != obs[after])[np.isin(obs[after], [0, 1])].mean()
nov = novelty(Z); dn = abs(nov[after].mean() - nov[before].mean()) / nov[before].std()
acheck('U5_label_shift_seen_by_error_not_novelty', err_a - err_b > 0.8 and dn < 0.1, f'err {err_b:.2f}->{err_a:.2f}, novelty shift={dn:.3f} sd')

# U6 The epsilon-mixture bound holds against a confidently wrong expert.
eps = 0.05; K = 20; N = 1000
y6 = rng.integers(0, K, N)
pA = rng.dirichlet(np.ones(K), N); pA[np.arange(N), y6] += 0.5; pA /= pA.sum(1, keepdims=True)
wrong = np.full((N, K), 1e-12); wrong[np.arange(N), (y6 + 1) % K] = 1; wrong /= wrong.sum(1, keepdims=True)
p = eps * pA + (1 - eps) * wrong
lhs = -np.log(p[np.arange(N), y6]); rhs = -np.log(pA[np.arange(N), y6]) - np.log(eps)
acheck('U6_eps_mixture_bound', np.all(lhs <= rhs + 1e-12) and -np.log(wrong[np.arange(N), y6]).mean() > 20,
       f'max(lhs-rhs)={np.max(lhs-rhs):.2e}, wrong-expert loss={-np.log(wrong[np.arange(N), y6]).mean():.1f}')
print('AUDIT FAILS:', afails)
if afails: sys.exit(1)

In [ ]:
def run_check(script):
    r = subprocess.run([sys.executable, script], capture_output=True, text=True)
    print(f'===== {script} =====\n' + r.stdout + (r.stderr[-3000:] if r.returncode else ''))
    if r.returncode != 0:
        raise SystemExit(f'{script} FAILED: stop here and send me this output.')
for s in ('test_lib.py', 'test_analysis.py', 'sk_check.py', 'audit_tests.py'):
    run_check(s)
print('All library, end-to-end, scikit-learn and audit checks passed.')

### GPU eigendecomposition (speed only). Used only if it gives the same ridge answers as numpy.

In [ ]:
def gpu_eigh(A):
    ev, V = torch.linalg.eigh(torch.from_numpy(np.ascontiguousarray(A)).to('cuda', torch.float64))
    return ev.cpu().numpy(), V.cpu().numpy()
atlas_lib.EIGH = np.linalg.eigh
if torch.cuda.is_available():
    rng = np.random.default_rng(0)
    X = rng.normal(size=(2000, 1152)); Y = rng.normal(size=(2000, 128))
    t = time.time(); B_cpu = atlas_lib.ridge_cv(X, Y, np.random.default_rng(1), split=1024)[0]; t_cpu = time.time() - t
    atlas_lib.EIGH = gpu_eigh
    t = time.time(); B_gpu = atlas_lib.ridge_cv(X, Y, np.random.default_rng(1), split=1024)[0]; t_gpu = time.time() - t
    same = np.allclose(B_cpu, B_gpu, rtol=1e-6, atol=1e-9 * np.abs(B_cpu).max())
    if not (same and t_gpu < t_cpu):
        atlas_lib.EIGH = np.linalg.eigh
    print(f'cpu {t_cpu:.1f}s, gpu {t_gpu:.1f}s, identical={same} -> using', 'GPU' if atlas_lib.EIGH is gpu_eigh else 'CPU')

## 3. Data and smoke test (small, fast; also checks determinism)

In [ ]:
data = load_cifar100()
SMOKE = dict(n_classes=15, n_tasks=3, M=200, reserve_per_class=10, lr=0.03, batch=10, stream_per_class=60)
D = run_one(0, SMOKE, data)
o = run_analysis(D, 0)
assert o['err_all']['O_full'] == 0
assert all(np.isfinite(v) and 0 <= v <= 100 for v in o['acc'].values())
assert all(np.isfinite(v) for v in o['err_all'].values())
D2 = run_one(0, SMOKE, data, log=lambda *a: None)
dmax = float(np.abs(D['F'][-1] - D2['F'][-1]).max())
print('determinism: max feature difference between two identical runs =', dmax,
      '(OK)' if dmax < 1e-3 else '(WARNING: not bit-reproducible on this GPU; results are still valid as a 5-seed statistic)')
print('smoke accuracies:', {k: round(v, 1) for k, v in o['acc'].items()})
del D, D2; gc.collect()

## 4. The experiment (2 configs × 5 seeds). Saved after every run; re-running skips finished runs.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
BASE = dict(n_classes=100, n_tasks=20, lr=0.03, batch=10, stream_per_class=None)
CONFIGS = {'A': dict(BASE, M=2000, reserve_per_class=20),
           'B': dict(BASE, M=1000, reserve_per_class=10)}
RESULTS_PATH = 'atlas_v2_results_partial.json'
results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
t_all = time.time()
for cname, cfg in CONFIGS.items():
    results.setdefault(cname, {})
    for seed in SEEDS:
        if str(seed) in results[cname]:
            print(f'{cname} seed {seed}: already done, skipping'); continue
        print(f'{cname} seed {seed}: running ...')
        D = run_one(seed, cfg, data)
        out = run_analysis(D, seed)
        out['config'] = cfg
        results[cname][str(seed)] = out
        del D; gc.collect()
        json.dump(results, open(RESULTS_PATH, 'w'))
        print(f'   analysis {out["seconds"]:.0f}s | O_full {out["acc"]["O_full"]:.1f} | CH_cv_buf {out["acc"]["CH_cv_buf"]:.1f}'
              f' | GR_multi_res {out["acc"]["GR_multi_res"]:.1f} | RC_LDA {out["acc"]["RC_LDA"]:.1f} | ER_head {out["acc"]["ER_head"]:.1f}'
              f' | elapsed {(time.time()-t_all)/60:.0f} min')

## 5. Results and the pre-registered comparisons

In [ ]:
summary = summarize(results)
final = dict(meta=dict(torch=torch.__version__, numpy=np.__version__, gpu=GPU, seeds=SEEDS, configs=CONFIGS,
                       eigh='gpu' if atlas_lib.EIGH is not np.linalg.eigh else 'cpu', finished=time.strftime('%Y-%m-%d %H:%M:%S')),
             summary=summary, runs=results)
json.dump(final, open('atlas_v2_results.json', 'w'), indent=1)
try:
    from google.colab import files; files.download('atlas_v2_results.json')
except Exception as e:
    print('Download it from the Files panel on the left (atlas_v2_results.json).', e)